In [2]:
import os
import numpy as np
import pandas as pd
import rasterio
from collections import Counter
import plotly.graph_objects as go
import webbrowser

# ============================================================
# INPUTS (EDIT)
# ============================================================
rasters = [
    ("00", r"D:\Khalil\PHD_doc\software\GeoSOS-FLUS\Python\Newcode\Class\c2000.tif"),
    ("10", r"D:\Khalil\PHD_doc\software\GeoSOS-FLUS\Python\Newcode\Class\c2010.tif"),
    ("20", r"D:\Khalil\PHD_doc\software\GeoSOS-FLUS\Python\Newcode\Class\c2020.tif"),
    ("50", r"D:\Khalil\PHD_doc\software\GeoSOS-FLUS\Python\Newcode\Class\DYNA_CLUE\CLUE_2050_INT.tif"),
]

classes = np.array([0,1,2,3,4,5,6,7,8,9,10], dtype=np.int16)

# If your rasters truly use -9999, keep this. Otherwise set to None.
FORCE_NODATA = -9999

# Filter tiny flows (pixels). Increase (e.g., 500, 1000) to reduce spaghetti.
MIN_FLOW_PIXELS = 200

# Output HTML (interactive)
out_html = r"D:\Khalil\PHD_doc\software\GeoSOS-FLUS\Python\Newcode\Class\DYNA_CLUE\lulc_sankey.html"

# Optional: convert pixel counts to area
# If you want area: set PIXEL_SIZE_M to your pixel size (e.g., 30 for Landsat).
# If None, values stay as pixel counts.
PIXEL_SIZE_M = None  # e.g., 30

# ============================================================
# CLASS NAMES + COLORS
# ============================================================
land_use_class_names = {
    0: "Building area",
    1: "Grassland",
    2: "Deciduous forest",
    3: "Coniferous forest",
    4: "Mixed forest",
    5: "Transitional forest",
    6: "Clearings",
    7: "Dwarf pine",
    8: "Peatland",
    9: "Rocks, stone seas",
    10: "Water bodies",
}

class_colors = {
    0: "#8c564b",
    1: "#ff7f0e",
    2: "#bcbd22",
    3: "#2ca02c",
    4: "#98df8a",
    5: "#7f7f7f",
    6: "#c7c7c7",
    7: "#17becf",
    8: "#e377c2",
    9: "#9467bd",
    10: "#1f77b4",
}

# ============================================================
# HELPERS
# ============================================================
def read_lulc(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.int32)
        nd = src.nodata
        tr = src.transform
        crs = src.crs
        shape = (src.height, src.width)
    return arr, nd, tr, crs, shape

def value_scale_pixels_to_area(value_pixels):
    """Convert pixels to hectares if PIXEL_SIZE_M is set; else return pixels."""
    if PIXEL_SIZE_M is None:
        return float(value_pixels)
    # hectares = pixels * (m*m) / 10,000
    return float(value_pixels) * (PIXEL_SIZE_M * PIXEL_SIZE_M) / 10000.0

def build_links_from_two_arrays(A, B, year_a, year_b, classes, nodata_a, nodata_b, min_flow):
    mask = np.isin(A, classes) & np.isin(B, classes)
    if nodata_a is not None:
        mask &= (A != nodata_a)
    if nodata_b is not None:
        mask &= (B != nodata_b)

    a = A[mask]
    b = B[mask]
    pairs = Counter(zip(a.tolist(), b.tolist()))

    rows = []
    for (ca, cb), v in pairs.items():
        if v < min_flow:
            continue
        rows.append({
            "source": f"{ca}-{year_a}",
            "target": f"{cb}-{year_b}",
            "value": value_scale_pixels_to_area(v),  # pixels or ha
        })
    return pd.DataFrame(rows)

def sankey_plot(df_links, title):
    # nodes
    nodes = pd.Index(pd.unique(df_links[["source", "target"]].values.ravel("K")))
    node_id = {n: i for i, n in enumerate(nodes)}

    def node_label(node):
        c_str, yr = node.split("-")
        c = int(c_str)
        name = land_use_class_names.get(c, f"class {c}")
        return f"{name} ({c})-{yr}"

    def node_color(node):
        c = int(node.split("-")[0])
        return class_colors.get(c, "#999999")

    node_labels = [node_label(n) for n in nodes]
    node_colors = [node_color(n) for n in nodes]

    sources = df_links["source"].map(node_id).astype(int).tolist()
    targets = df_links["target"].map(node_id).astype(int).tolist()
    values  = df_links["value"].astype(float).tolist()

    fig = go.Figure(
        data=[
            go.Sankey(
                arrangement="snap",
                node=dict(
                    pad=18,
                    thickness=14,
                    line=dict(color="rgba(0,0,0,0.25)", width=0.6),
                    label=node_labels,
                    color=node_colors,
                ),
                link=dict(
                    source=sources,
                    target=targets,
                    value=values,
                    color="rgba(120,120,120,0.25)",
                ),
            )
        ]
    )

    unit = "pixels" if PIXEL_SIZE_M is None else "ha"
    fig.update_layout(
        title=f"{title} (values in {unit}, min flow={MIN_FLOW_PIXELS})",
        font=dict(size=12),
        height=650,
        width=1200,
        margin=dict(l=20, r=20, t=60, b=20),
    )
    return fig

# ============================================================
# RUN
# ============================================================
arrays = []
nodatas = []
meta = []

for yr, p in rasters:
    arr, nd, tr, crs, shape = read_lulc(p)
    arrays.append(arr)
    nodatas.append(nd)
    meta.append((yr, p, tr, crs, shape))

# Force nodata if you want
if FORCE_NODATA is not None:
    nodatas = [FORCE_NODATA] * len(nodatas)

# Basic alignment sanity check (does NOT reproject; it warns you)
for i in range(1, len(meta)):
    if meta[i][4] != meta[0][4]:
        print("WARNING: raster shapes differ:", meta[0][4], "vs", meta[i][4])
    if meta[i][2] != meta[0][2]:
        print("WARNING: raster transforms differ (grids not aligned). Transitions may be invalid.")
    if meta[i][3] != meta[0][3]:
        print("WARNING: raster CRS differ. Transitions may be invalid.")

# Build links for consecutive years
dfs = []
for i in range(len(rasters) - 1):
    year_a, _ = rasters[i]
    year_b, _ = rasters[i + 1]

    df = build_links_from_two_arrays(
        arrays[i], arrays[i + 1],
        year_a=year_a, year_b=year_b,
        classes=classes,
        nodata_a=nodatas[i],
        nodata_b=nodatas[i + 1],
        min_flow=MIN_FLOW_PIXELS
    )
    dfs.append(df)

df_links = pd.concat(dfs, ignore_index=True)

if df_links.empty:
    raise ValueError("No flows left after filtering. Reduce MIN_FLOW_PIXELS (e.g., 50) or check class codes/nodata.")

print("Flows kept:", len(df_links))
print("Total value represented:", df_links["value"].sum())

fig = sankey_plot(df_links, title="Land use/cover dynamic changes")

# IMPORTANT:
# Do NOT call fig.show() (it triggers nbformat error in non-notebook envs).
# Save HTML and open it in your browser.
os.makedirs(os.path.dirname(out_html), exist_ok=True)
fig.write_html(out_html)
print("Saved Sankey HTML:", out_html)

# Auto-open in default browser
webbrowser.open("file:///" + out_html.replace("\\", "/"))


Flows kept: 89
Total value represented: 304959.0
Saved Sankey HTML: D:\Khalil\PHD_doc\software\GeoSOS-FLUS\Python\Newcode\Class\DYNA_CLUE\lulc_sankey.html


True